In [1]:
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from scipy.linalg import svd
from scipy.fft import fft, ifft, fftfreq
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 20
CONTEXT_LEN = 512
PRED_LEN    = 96
DATA_DIR    = './ts_data'

Device: cpu


In [2]:
import inspect
from panda.patchtst.pipeline import PatchTSTPipeline
print(inspect.signature(PatchTSTPipeline.from_pretrained))

(mode: str, pretrain_path: str, **kwargs)


In [3]:
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline

# Published checkpoint
print('Loading published Panda...')
panda_published = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

# Retrained baseline
print('Loading retrained baseline...')
panda_baseline = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='./checkpoint-50k-baseline',
    device_map=device,
)

# Koopman ablation
print('Loading Koopman ablation...')
panda_koopman = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='./checkpoint-50k-koopman_ablation',
    device_map=device,
)

print('All models loaded.')

Loading published Panda...
Loading retrained baseline...
Loading Koopman ablation...
All models loaded.


In [4]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def instance_norm_window(x_CT):
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std(axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

def make_forecast_fn(pipeline):
    """Returns a panda_forecast-compatible function for any PatchTSTPipeline."""
    def forecast(context_np, horizon):
        TRAIN_H   = 128
        remaining = horizon
        ctx       = context_np.copy()
        preds     = []
        while remaining > 0:
            h         = min(TRAIN_H, remaining)
            context_t = torch.tensor(ctx.T, dtype=torch.float32)
            with torch.no_grad():
                pred = pipeline.predict(
                    context_t, h,
                    limit_prediction_length=False,
                    sliding_context=True,
                )
            p = pred.squeeze().cpu().numpy()
            if p.ndim == 1:
                p = p[:, None]
            if p.shape[0] != context_np.shape[0]:
                p = p.T
            preds.append(p[:, :h])
            ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
            remaining -= h
        return np.concatenate(preds, axis=1)
    return forecast

def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='model_a', name_b='model_b'):
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv   = np.median(mae_b) - np.median(mae_a)
    iqr_a = np.percentile(mae_a, 75) - np.percentile(mae_a, 25)
    iqr_b = np.percentile(mae_b, 75) - np.percentile(mae_b, 25)
    sig   = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')

    result = {
        'label'           : label,
        'horizon'         : horizon,
        'name_a'          : name_a,
        'name_b'          : name_b,
        f'{name_a}_mae'   : np.median(mae_a),
        f'{name_a}_iqr'   : iqr_a,
        f'{name_b}_mae'   : np.median(mae_b),
        f'{name_b}_iqr'   : iqr_b,
        'advantage_mae'   : adv,
        'wilcoxon_p'      : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}  '
        f'{name_b}={np.median(mae_b):.4f}  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Helpers defined.')

Helpers defined.


In [5]:
data_weather = load_ts(f'{DATA_DIR}/weather.csv')
print(f'Weather: {data_weather.shape}')

Weather: (21, 52696)


In [6]:
from scipy.integrate import solve_ivp

def simulate_burgers_stable(T=1000, N_x=128, nu=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x
    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub
    k         = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias   = np.abs(k) <= N_x // 3
    L_op      = -nu * k**2
    u0_hat    = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias
    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin
    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4    = rhs_hat(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
    return U

def pca_reduction(U, n_components):
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)

print('Burgers helpers defined.')

Burgers helpers defined.


In [7]:
fn_published = make_forecast_fn(panda_published)
fn_baseline  = make_forecast_fn(panda_baseline)
fn_koopman   = make_forecast_fn(panda_koopman)

ablation_results = []

# Weather
print('=== Weather ===')
for h in [96, 192, 336]:
    for fn, tag in [(fn_published, 'published'),
                    (fn_baseline,  'retrained_base'),
                    (fn_koopman,   'koopman_ablation')]:
        r = evaluate(data_weather, h, n_windows=N_WINDOWS,
                     label=f'Weather_{tag}_H{h}',
                     fn_a=fn, fn_b=fn_published,  # compare each vs published
                     name_a=tag, name_b='published')
        if r:
            r['dataset']   = 'Weather'
            r['model_tag'] = tag
            ablation_results.append(r)

# Burgers nu=1.0
print('\n=== Burgers nu=1.0 ===')
U_burgers    = simulate_burgers_stable(T=1500, N_x=128, nu=1.0)
pca_data     = pca_reduction(U_burgers, 16)
data_burgers = pca_data.T

for h in [96, 192, 336]:
    for fn, tag in [(fn_published, 'published'),
                    (fn_baseline,  'retrained_base'),
                    (fn_koopman,   'koopman_ablation')]:
        r = evaluate(data_burgers, h, n_windows=N_WINDOWS,
                     label=f'Burgers_{tag}_H{h}',
                     fn_a=fn, fn_b=fn_published,
                     name_a=tag, name_b='published')
        if r:
            r['dataset']   = 'Burgers_nu1.0'
            r['model_tag'] = tag
            ablation_results.append(r)

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv('koopman_ablation_results.csv', index=False)
print('\nSaved koopman_ablation_results.csv')
print(df_ablation[['dataset','model_tag','horizon','advantage_mae','wilcoxon_p']].to_string())

=== Weather ===
  Weather_published_H96                               H=  96  published=0.6352  published=0.6352  Adv=+0.0000  p=1.000
  Weather_retrained_base_H96                          H=  96  retrained_base=0.8238  published=0.6352  Adv=-0.1886  p=0.997
  Weather_koopman_ablation_H96                        H=  96  koopman_ablation=0.6569  published=0.6352  Adv=-0.0216  p=0.918
  Weather_published_H192                              H= 192  published=0.7229  published=0.7229  Adv=+0.0000  p=1.000
  Weather_retrained_base_H192                         H= 192  retrained_base=0.8685  published=0.7229  Adv=-0.1455  p=0.955
  Weather_koopman_ablation_H192                       H= 192  koopman_ablation=0.8692  published=0.7229  Adv=-0.1463  p=0.993
  Weather_published_H336                              H= 336  published=0.8284  published=0.8284  Adv=+0.0000  p=1.000
  Weather_retrained_base_H336                         H= 336  retrained_base=0.9291  published=0.8284  Adv=-0.1008  p=0.992
  W

In [8]:
from scipy.integrate import solve_ivp

def simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED):
    """Simple harmonic oscillator x'' + omega^2 x = 0."""
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    series = np.array(traj, dtype=np.float32)
    return series[500:]  # discard transient

harmonic_series = simulate_harmonic(n_steps=4000, omega=1.0)
data_harmonic   = harmonic_series[None, :]  # (1, T)
print(f'Harmonic series shape: {data_harmonic.shape}')
print(f'Length after transient: {data_harmonic.shape[1]} steps')

Harmonic series shape: (1, 3500)
Length after transient: 3500 steps


In [9]:
print('=== Harmonic Oscillator ===')
print('Scientific question: does Koopman ablation specifically lose the harmonic advantage?')
print('Theoretical basis: sinusoids are exact Koopman eigenfunctions for linear systems.')
print('-' * 70)

harmonic_results = []

for h in [96, 192, 336]:
    for fn, tag in [(fn_published,  'published'),
                    (fn_baseline,   'retrained_base'),
                    (fn_koopman,    'koopman_ablation')]:
        r = evaluate(data_harmonic, h, n_windows=20,
                     label=f'Harmonic_{tag}_H{h}',
                     fn_a=fn, fn_b=fn_published,
                     name_a=tag, name_b='published')
        if r:
            r['dataset']   = 'Harmonic'
            r['model_tag'] = tag
            harmonic_results.append(r)

df_harmonic = pd.DataFrame(harmonic_results)
df_harmonic.to_csv('koopman_ablation_harmonic.csv', index=False)
print('\nSaved koopman_ablation_harmonic.csv')

# Summary — focus on baseline vs ablation comparison
print('\n=== Key comparison: retrained_base vs koopman_ablation ===')
print(f'{"H":>5} | {"baseline_mae":>12} | {"ablation_mae":>12} | {"gap":>8} | interpretation')
print('-' * 65)
for h in [96, 192, 336]:
    base = df_harmonic[(df_harmonic.model_tag=='retrained_base') & 
                       (df_harmonic.horizon==h)]
    abl  = df_harmonic[(df_harmonic.model_tag=='koopman_ablation') & 
                       (df_harmonic.horizon==h)]
    if len(base) and len(abl):
        b_mae = float(base.iloc[0]['retrained_base_mae'])
        a_mae = float(abl.iloc[0]['koopman_ablation_mae'])
        gap   = a_mae - b_mae  # positive = ablation worse
        if gap > 0.05:
            interp = 'Ablation worse — Koopman lifting helps on harmonic'
        elif gap < -0.05:
            interp = 'Ablation better — Koopman lifting hurts on harmonic'
        else:
            interp = 'Similar — Koopman lifting neutral on harmonic'
        print(f'{h:>5} | {b_mae:>12.4f} | {a_mae:>12.4f} | {gap:>+8.4f} | {interp}')

=== Harmonic Oscillator ===
Scientific question: does Koopman ablation specifically lose the harmonic advantage?
Theoretical basis: sinusoids are exact Koopman eigenfunctions for linear systems.
----------------------------------------------------------------------
  Harmonic_published_H96                              H=  96  published=0.0688  published=0.0688  Adv=+0.0000  p=1.000
  Harmonic_retrained_base_H96                         H=  96  retrained_base=0.3483  published=0.0688  Adv=-0.2795  p=1.000
  Harmonic_koopman_ablation_H96                       H=  96  koopman_ablation=0.2050  published=0.0688  Adv=-0.1362  p=1.000
  Harmonic_published_H192                             H= 192  published=0.1182  published=0.1182  Adv=+0.0000  p=1.000
  Harmonic_retrained_base_H192                        H= 192  retrained_base=0.5292  published=0.1182  Adv=-0.4110  p=1.000
  Harmonic_koopman_ablation_H192                      H= 192  koopman_ablation=0.4767  published=0.1182  Adv=-0.3586  p=1.

In [10]:
def simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED):
    """Van der Pol oscillator: nonlinear limit cycle."""
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic  = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-8, atol=1e-8)
    series = sol.y[0].astype(np.float32)
    return series[500:]  # discard transient

def simulate_duffing(n_steps=4000, delta=0.3, alpha=-1.0,
                     beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    """Duffing oscillator: nonlinear, weakly chaotic."""
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t    = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax    = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    series = np.array(traj, dtype=np.float32)
    return series[500:]

# Simulate both
vdp_series     = simulate_vanderpol(n_steps=4000, mu=2.0)
duffing_series = simulate_duffing(n_steps=4000)

data_vdp     = vdp_series[None, :]
data_duffing = duffing_series[None, :]

print(f'Van der Pol shape: {data_vdp.shape}')
print(f'Duffing shape:     {data_duffing.shape}')

Van der Pol shape: (1, 3500)
Duffing shape:     (1, 3500)


In [11]:
print('=== Van der Pol (nonlinear limit cycle) ===')
print('HYP A predicts: ablation worse (nonlinear system)')
print('HYP B predicts: ablation better (periodic signal)')
print('-' * 70)

continuum_results = []

for sys_name, data_sys in [('VanderPol', data_vdp),
                            ('Duffing',   data_duffing)]:
    print(f'\n--- {sys_name} ---')
    for h in [96, 192, 336]:
        for fn, tag in [(fn_published, 'published'),
                        (fn_baseline,  'retrained_base'),
                        (fn_koopman,   'koopman_ablation')]:
            r = evaluate(data_sys, h, n_windows=20,
                         label=f'{sys_name}_{tag}_H{h}',
                         fn_a=fn, fn_b=fn_published,
                         name_a=tag, name_b='published')
            if r:
                r['dataset']   = sys_name
                r['model_tag'] = tag
                continuum_results.append(r)

df_continuum = pd.DataFrame(continuum_results)
df_continuum.to_csv('koopman_ablation_continuum.csv', index=False)
print('\nSaved koopman_ablation_continuum.csv')

# Summary — baseline vs ablation for each system
print('\n=== Key comparison: retrained_base vs koopman_ablation ===')
print(f'{"system":>12} | {"H":>5} | {"baseline":>10} | {"ablation":>10} | {"gap":>8} | verdict')
print('-' * 75)

for sys_name in ['VanderPol', 'Duffing']:
    for h in [96, 192, 336]:
        base_rows = df_continuum[(df_continuum.dataset==sys_name) &
                                  (df_continuum.model_tag=='retrained_base') &
                                  (df_continuum.horizon==h)]
        abl_rows  = df_continuum[(df_continuum.dataset==sys_name) &
                                  (df_continuum.model_tag=='koopman_ablation') &
                                  (df_continuum.horizon==h)]
        if len(base_rows) and len(abl_rows):
            # get the model MAE column dynamically
            base_col = [c for c in base_rows.columns if 'retrained_base' in c and 'mae' in c and 'iqr' not in c]
            abl_col  = [c for c in abl_rows.columns  if 'koopman' in c and 'mae' in c and 'iqr' not in c]
            if base_col and abl_col:
                b_mae = float(base_rows.iloc[0][base_col[0]])
                a_mae = float(abl_rows.iloc[0][abl_col[0]])
                gap   = a_mae - b_mae  # positive = ablation worse
                if gap > 0.03:
                    verdict = 'ablation WORSE  → supports HYP A'
                elif gap < -0.03:
                    verdict = 'ablation BETTER → supports HYP B'
                else:
                    verdict = 'similar         → inconclusive'
                print(f'{sys_name:>12} | {h:>5} | {b_mae:>10.4f} | {a_mae:>10.4f} | {gap:>+8.4f} | {verdict}')

print('\n=== Summary across all systems ===')
print('System        | Linear? | Periodic? | Ablation vs Baseline | Supports')
print('-' * 70)
print('Harmonic      |   Yes   |    Yes    | Better (all H)       | HYP A or B')
print('Van der Pol   |   No    |    Yes    | ??? (see above)      | diagnostic')
print('Duffing       |   No    | Quasi     | ??? (see above)      | diagnostic')
print('Burgers nu1.0 |   No    |    No     | Worse (all H)        | HYP A')

=== Van der Pol (nonlinear limit cycle) ===
HYP A predicts: ablation worse (nonlinear system)
HYP B predicts: ablation better (periodic signal)
----------------------------------------------------------------------

--- VanderPol ---
  VanderPol_published_H96                             H=  96  published=0.0368  published=0.0368  Adv=+0.0000  p=1.000
  VanderPol_retrained_base_H96                        H=  96  retrained_base=0.0982  published=0.0368  Adv=-0.0615  p=1.000
  VanderPol_koopman_ablation_H96                      H=  96  koopman_ablation=0.1063  published=0.0368  Adv=-0.0696  p=1.000
  VanderPol_published_H192                            H= 192  published=0.0511  published=0.0511  Adv=+0.0000  p=1.000
  VanderPol_retrained_base_H192                       H= 192  retrained_base=0.1961  published=0.0511  Adv=-0.1450  p=1.000
  VanderPol_koopman_ablation_H192                     H= 192  koopman_ablation=0.1713  published=0.0511  Adv=-0.1202  p=1.000
  VanderPol_published_H336  

In [13]:
# Sanity check: both models on a simple known system
# Use Lorenz which we know Panda handles well

from scipy.integrate import solve_ivp

def simulate_lorenz(n_steps=3000, rho=28.0, seed=42):
    rng = np.random.default_rng(seed)
    ic  = rng.standard_normal(3).tolist()
    t_span = (0, n_steps*0.01)
    t_eval = np.linspace(*t_span, n_steps)
    sol = solve_ivp(lambda t,y: [10*(y[1]-y[0]), y[0]*(rho-y[2])-y[1], y[0]*y[1]-8/3*y[2]],
                    t_span, ic, t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y[0].astype(np.float32)

lorenz = simulate_lorenz(n_steps=3000, rho=28.0)
data_lorenz = lorenz[None, :]  # (1, T)

print('In-distribution sanity check (Lorenz rho=28):')
for fn, name in [(fn_baseline, 'retrained_baseline'),
                  (fn_koopman,  'koopman_ablation'),
                  (fn_published, 'published')]:
    C, T = data_lorenz.shape
    ctx_raw = data_lorenz[:, :512]
    tgt_raw = data_lorenz[:, 512:512+96]
    ctx_norm, mu, std = instance_norm_window(ctx_raw)
    tgt_norm = (tgt_raw - mu) / std
    pred = fn(ctx_norm, 96)
    mae_val = np.mean(np.abs(pred - tgt_norm))
    print(f'  {name}: MAE = {mae_val:.4f}')

In-distribution sanity check (Lorenz rho=28):
  retrained_baseline: MAE = 0.2750
  koopman_ablation: MAE = 0.6171
  published: MAE = 0.0208
